# 1. Advanced RAG란 무엇인가

**RAG**는 LLM(대규모 언어 모델)이 답을 만들 때, **외부 지식**(**문서/DB/웹 등**)**을 검색(Retrieval)해서 근거(Context)로 붙인 뒤 생성**(**Generation**)하는 구조이다. 그런데 기본 RAG(Naive RAG)는 실무에서 다음 문제가 있다.

- 검색이 "비슷한 것"은 찾는데 "정답 근거"를 제대로 못 찾는 경우
- 근거가 있어도 LLM이 **hallucination**(**환각**)를 섞거나, 근거와 다른 답을 하는 경우
- 검색한 문서가 너무 길고 복잡해서 **정답에 필요한 부분**(**chunk**)을 LLM이 제대로 찾지 못해 답변 품질이 안좋은 경우

**Advanced RAG**는 이런 "현업형 문제"를 줄이기 위해 **데이터베이스 구축 단계, 검색 전단계, 검색 후 단계, 생성 단계에서 고도화**하는 설계 패턴이다.
기본 RAG(Naive RAG)가 "사람이 대충 찾아준 참고자료로 글 쓰는 것"이라면, Advanced RAG는 "사서(검색) + 편집자(정제) + 팩트체커(검증) + 작가(생성)가 협업하는 파이프라인"에 가깝다.

## 종류

### 검색 품질을 올리는 고급 Retrieval

- **하이브리드 검색**: 키워드(BM25) + 벡터(임베딩) 조합
- **멀티-쿼리/쿼리 재작성(Query rewriting)**: 질문을 더 잘 검색되게 바꿔 검색 성공률을 올림
- **메타데이터 필터링**: 메타데이터 필터링을 통해 범위를 좁혀 **정확도**를 올림. 의미기반 검색과 키워드 검색을 조합한 효과.
- **리랭킹(Re-ranking)**: 후보 문서를 많이 가져온 뒤, "정답에 가까운 순서"로 다시 정렬

### Chunking/인덱싱 전략 고도화
- 벡터 데이터 베이스 구축시 어떤 구조로 문서를 분할하고 Indexing 할지의 전략
  - **구조 기반 분할**: 헤더/섹션/표/코드블록
  - **계층형 인덱스**: 문서 요약 → 섹션 → 세부 chunk, Parent-Child 구조
  - **윈도우 확장**: 필요 시 앞 뒤 문맥을 함께 붙임
  
### 검색 결과의 "정제/조립" 단계 추가

- 중복 제거, 노이즈 제거, 관련 부분만 발췌
- 여러 chunk를 **질문 관점으로 재구성**
  → LLM에 그대로 던지는 게 아니라 "답변에 쓰기 좋은 근거 묶음"으로 편집하여 전달한다.

### 생성 단계의 신뢰성 강화(가드레일)

- **근거 기반 답변 강제**: 검색한 문서에 답변의 근거가 없으면 "답을 모른다."고 답변하도록 프롬프트를 구성한다.
- **인용/근거 스니펫 포함**: 어느 문서에서 답이 나왔는지 근거를 보여주도록 하여 답변의 신뢰성을 높인다.
- **자기검증(Verification)**: 답을 만든 뒤 "근거와 모순 없는지" 재확인 하도록 한다.
- **불확실성 처리**: 애매하면 추가 질문(Clarification)을 하거나 또는 답변 범위 제한한다.


## Advanced RAG가 필요한 일반적인 경우

- 문서가 많고(수천~수백만), 구조가 복잡한 경우(매뉴얼/규정/기술문서)
- 질문이 단순 검색이 아니라 "비교/조건/절차/근거 요구"가 많은 복잡한 질문인 경우.
- 최신성/정확성이 중요한 경우(정책, 금융, 의료, 보안, 운영 장애 대응)
- "대충 그럴듯한 답"이 아니라 **근거가 있는 답**이 필수인 경우


In [1]:
###############################################################
# VectorStore, Retriever 준비
###############################################################
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

from langchain_openai import ChatOpenAI
from langchain_qdrant import FastEmbedSparse, QdrantVectorStore, RetrievalMode
from qdrant_client import QdrantClient, models
from qdrant_client.models import Distance, SparseVectorParams, VectorParams
from langchain_openai import OpenAIEmbeddings

In [5]:
# ##############################################################
# 데이터 준비
##############################################################

def load_and_split_olympic_data(file_path="data/olympic_wiki.md"):
    with open(file_path, "r", encoding="utf-8") as fr:
        olympic_text = fr.read()

    # Split
    splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on=[
            ("#", "Header 1"),
            ("##", "Header 2"),
            ("###", "Header 3"),
        ],
        # strip_headers=False, # 문서에 header 포함 여부(default: True - 제거)
    )

    return splitter.split_text(olympic_text)

In [ ]:
#################################################################
# Vector DB 연결
# retriever 생성
#################################################################

# collection 삭제후 생성 (데이터 넣지는 않음)
def get_vectorstore(collection_name: str = "olympic_info_wiki"):

    #######################################
    # Qdrant Collection 생성 (sparse + dense)
    #######################################
    dense_embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
    sparse_embeddings = FastEmbedSparse(model_name="Qdrant/bm25")

    client = QdrantClient(host="localhost", port=6333)

    #삭제후 생성
    if client.collection_exists(collection_name):
        result = client.delete_collection(collection_name=collection_name)

    client.create_collection(
        collection_name=collection_name,
        vectors_config={"dense": VectorParams(size=3072, distance=Distance.COSINE)},
        sparse_vectors_config={ 
            "sparse": SparseVectorParams()
        },
    )

    ######################################
    # VectorStore 생성 (Hybrid 모드)
    ######################################
    vector_store = QdrantVectorStore(
        client=client,
        collection_name=collection_name,
    
        embedding=dense_embeddings,
        
        sparse_embedding=sparse_embeddings,
        retrieval_mode=RetrievalMode.HYBRID,
    
        vector_name="dense",
        sparse_vector_name="sparse",
    )
    
    ######################################
    # Document들 추가
    ######################################
    documents = load_and_split_olympic_data()
    vector_store.add_documents(documents=documents)

    return vector_store


def get_retriever(vector_store, k: int = 5):
    retriever = vector_store.as_retriever(
        search_kwargs={"k": k}
    )
    return retriever

In [8]:
vectorstore = get_vectorstore()
basic_retriever = get_retriever(vectorstore, k=7)

In [9]:
basic_retriever.invoke("근대 올림픽은 언제 시작했나요?")

[Document(metadata={'Header 1': '올림픽', '_id': 'ce130d8d-fe5d-4b41-b314-b70514ab94b2', '_collection_name': 'olympic_info_wiki'}, page_content='올림픽(영어: Olympic Games, 프랑스어: Jeux olympiques)은 전 세계 각 대륙 각국에서 모인 수천 명의 선수가 참가해 여름과 겨울에 스포츠 경기를 하는 국제적인 대회이다. 전 세계에서 가장 큰 지구촌 최대의 스포츠 축제인 올림픽은 세계에서 가장 인지도있는 국제 행사이다. 올림픽은 2년마다 하계 올림픽과 동계 올림픽이 번갈아 열리며, 국제 올림픽 위원회(IOC)가 감독하고 있다. 또한 오늘날의 올림픽은 기원전 8세기부터 서기 5세기에 이르기까지 고대 그리스 올림피아에서 열렸던 올림피아 제전에서 비롯되었다. 그리고 19세기 말에 피에르 드 쿠베르탱 남작이 고대 올림피아 제전에서 영감을 얻어, 근대 올림픽을 부활시켰다. 이를 위해 쿠베르탱 남작은 1894년에 IOC를 창설했으며, 2년 뒤인 1896년에 그리스 아테네에서 제 1회 올림픽이 열렸다. 이때부터 IOC는 올림픽 운동의 감독 기구가 되었으며, 조직과 활동은 올림픽 헌장을 따른다. 오늘날 전 세계 대부분의 국가에서 올림픽 메달은 매우 큰 영예이며, 특히 올림픽 금메달리스트는 국가 영웅급의 대우를 받으며 스포츠 스타가 된다. 국가별로 올림픽 메달리스트들에게 지급하는 포상금도 크다. 대부분의 인기있는 종목들이나 일상에서 쉽게 접하고 즐길 수 있는 생활스포츠 종목들이 올림픽이라는 한 대회에서 동시에 열리고, 전 세계 대부분의 국가 출신의 선수들이 참여하는 만큼 전 세계 스포츠 팬들이 가장 많이 시청하는 이벤트이다. 2008 베이징 올림픽의 모든 종목 누적 시청자 수만 47억 명에 달하며, 이는 인류 역사상 가장 많은 수의 인구가 시청한 이벤트였다.  \n또한 20세기에 올림픽 운동이 발전함에 따라, IOC는 변화하는 세계의 사회 환경에 적응해

# Rerank

## 개념

- RAG의 정확도는 관련 정보의 컨텍스트 내 존재 유무가 아니라 순서가 중요하다. 즉, 관련 정보가 컨텍스트 내 상위권에 위치하고 있을 때 좋은 답변을 얻을 수 있다는 뜻이다. 
- **Rerank**는 RAG 시스템에서 **초기 검색 단계에서 추출된 후보 문서들의 순위를 재조정**하는 기법이다. 
- 벡터 유사도 기반의 빠른 1차 검색 후, 보다 정밀한 모델(예: Cross-encoder, LLM 등)을 활용해 질문과 검색된 문서간의 의미론적 관련성을 평가하여, 실제로 답변 생성에 가장 **적합한 문서들이 상위**에 오도록 순서를 다시 매긴다. 이를 통해 LLM이 더 정확하고 관련성 높은 정보를 바탕으로 답변을 생성할 수 있게 도와준다.

## 방법

- **Cross-encoder 기반 Rerank**  
  - Cross Encoder를 이용해서 순위를 재 지정한다.
  - Cross-encoder
    - 질문과 문서를 같이 입력으로 받아 둘간의 유사도 점수를 예측하도록 학습한 모델.
    - 학습을 두 문장의 유사도록 예측하도록 학습하였기 때문에 단순 유사도 검사 보다 두 문장간의 의미적 관련성등을 이용해 유사도를 예측하기 때문에 더 정확한 결과를 보인다.
  - 1차적으로 검색한 문서와 질문간의 유사도를 **cross-encoder**로 다시 계산해서 문서의 순위를 재 조정한다.
  
  > - **Bi-encoder**
  >     - 질문 (query)와 문서(document)를 각각 독립적으로 인코딩한 후, 벡터 간 유사도 계산
  >     - Encoder 모델은 개별 문장을 입력받아 embedding vector를 출력한다. 질문과 문서를 각각 encoding한 뒤에 둘 간의 유사도를 계산한다.
  
  ![bi_crosss_encoder](figures/bi_cross_encoder.png)

  \[출처:https://aws.amazon.com/ko/blogs/tech/korean-reranker-rag/\]

- **LLM 기반 Rerank**  
  GPT-3, GPT-4 등 LLM을 활용해 각 문서가 질문에 얼마나 부합하는지 평가하여 순위를 매긴다. 성능이 뛰어나지만 비용이 높고 응답 속도가 느릴 수 있다.
## Rerank RAG 프로세스  
1. 1차 검색(예: 임베딩 기반 벡터 검색)으로 상위 k개 문서 추출.  
2. Reranker 모델에 질문-문서 쌍을 입력.  
3. 각 쌍의 관련성 점수 산출 및 재정렬.  
4. 상위 n개 문서를 LLM의 컨텍스트로 전달하여 답변 생성.

## 장단점

- **장점**  
  - Rerank를 적용하면 단순 벡터 유사도 기반 검색보다 훨씬 정교하게 질문과 관련된 정보를 추출할 수 있다. 
  - 실제로 생성되는 답변의 품질이 크게 향상되며, 도메인 특화 정보나 복잡한 질의에도 높은 정확도를 보인다.

- **단점**  
  - Cross-encoder나 LLM 기반 Rerank는 연산량이 많아 실시간 응답이 필요한 대규모 서비스에선 속도 저하가 발생할 수 있다.
  - 초기 검색 결과(상위 k개)에만 적용하므로, 1차 검색의 품질이 낮으면 Rerank 효과가 제한적일 수 있다.


## ContextualCompressionRetriever)와 ReRank

- RAG의 성능을 떨어뜨리는 한 가지 어려움은 검색된 문서들 중에 질의와 관련성 없는 문서가 포함되어 있을 수 있다는 것이다. 이런 것을 포함한 전체 문서를 LLM에 전달 하면 응답 성능이 나빠질 뿐 아니라 쓸데 없는 비용도 많이 들게 된다.
- 이를 해결하기 위한 목적으로 Contextual Compression(맥락적 압축) 방식이 고안되었다.
- 검색된 문서들을 그대로 반환하는 대신, 주어진 쿼리의 맥락을 사용해 그것들을 압축하여 관련 정보만 반환할 수 있도록 한다.
- 압축의 방법
  - **내용 압축**: 각 문서 조각에서 불필요한 내용을 제거하고 핵심 문장만 추출
  - **문서 필터링**: 전체 문서 중 질의와 관련 없는 문서 자체를 제거.
- ContextualCompressionRetriever는 검색기(base retriever), 문서 압축기(Document Compressor) 두 컴포넌트가 필요.
    - 기본 검색기: 쿼리와 관련된 문서를 검색한다. 이것이 초기 검색 결과 문서가 된다.
    - 문서 압축기: 초기 검색 문서 결과에서 질의와 관련 없는 내용을 제거해서 전체 문서의 양을 줄인다.(압축)
- **ReRank**는 Contextual Compression 방식 중 **문서 필터링 방식**의 알고리즘이다.
  - 문서 압축기로 ReRanker를 사용한다.
  - 기본 검색기에서 충분한 양의 문서를 검색 한 뒤 ReRanker를 이용해 순위를 재조정하고 그중 상위 K개를 최종 context로 사용한다.   

In [10]:
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

retriever = get_retriever(vectorstore, k=10)
reranker_model_id = "BAAI/bge-reranker-v2-m3"
reranker = HuggingFaceCrossEncoder(model_name=reranker_model_id)

config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

c:\Users\Playdata\SKN\SKN31\10_AI_Agent\.venv\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Playdata\.cache\huggingface\hub\models--BAAI--bge-reranker-v2-m3. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.17k [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

In [11]:
compressor = CrossEncoderReranker(model=reranker, top_n=5)
compressor_retriever = ContextualCompressionRetriever(
    base_retriever=retriever,
    base_compressor=compressor,
)

In [12]:
query = "올림픽에서 발생한 다양한 논란들에 대해서 정리해줘."
basic_result = retriever.invoke(query)
rerank_result = compressor_retriever.invoke(query)

In [13]:
basic_result

[Document(metadata={'Header 1': '올림픽', 'Header 2': '논란', 'Header 3': '정치', '_id': '39f94c6e-642d-4870-ab0b-4b8dc628d1ad', '_collection_name': 'olympic_info_wiki'}, page_content="쿠베르탱이 말했던 원래 이념과는 반대로 올림픽이 정치 혹은 체제 선전의 장으로 이용되는 경우가 있었다. 1936년 하계 올림픽을 개최할 때 당시의 나치독일은 나치는 자비롭고 평화를 위한다는 것을 설명하고 싶어했다. 또 이 올림픽에서 아리안족의 우월함을 보여줄 생각이었으나 이는 흑인이었던 제시 오언스가 금메달을 4개나 따내면서 실현되지는 못했다. 소련은 헬싱키에서 열린 1952년 하계 올림픽 때 처음으로 참가했다. 그 전에는 소련이 조직한 스파르타키아다라는 대회에 1928년부터 참가했었다. 다른 공산주의 국가들은 1920년대와 1930년대의 전쟁 기간 사이에 노동자 올림픽(Socialist Workers' Sport International)을 조직했는데, 이는 올림픽을 자본가와 귀족들의 대회로 여기고 그에 대한 대안으로 고안된 대회였다. 그 이후 소련은 1956년 하계 올림픽부터 1988년 하계 올림픽까지 엄청난 스포츠강국의 면모를 보여주며 올림픽에서의 명성을 드높였다.  \n선수 개인이 자신의 정치적 성향에 대해 표현하기도 했다. 멕시코 시티에서 열린 1968년 하계 올림픽의 육상부문 200m 경기에서 각각 1위와 3위를 한 미국의 토미 스미스와 존 카를로스는 시상식 때 블랙 파워 설루트(Black Power salute , 흑인 차별 반대 행위)를 선보였으며 2위를 한 피터 노먼도 상황을 깨닫고 스미스와 카를로스의 행위를 지지한다는 뜻에서 급하게 인권을 위한 올림픽 프로젝트(OPHR) 배지를 달았다. 이 사건에 대해서 IOC 위원장이었던 에이버리 브런디지는 미국 올림픽 위원회에 이 두 선수를 미국으로 돌려보내거나 미국 육상팀 전부를 돌려보내는 

In [14]:
rerank_result

[Document(metadata={'Header 1': '올림픽', '_id': 'ce130d8d-fe5d-4b41-b314-b70514ab94b2', '_collection_name': 'olympic_info_wiki'}, page_content='올림픽(영어: Olympic Games, 프랑스어: Jeux olympiques)은 전 세계 각 대륙 각국에서 모인 수천 명의 선수가 참가해 여름과 겨울에 스포츠 경기를 하는 국제적인 대회이다. 전 세계에서 가장 큰 지구촌 최대의 스포츠 축제인 올림픽은 세계에서 가장 인지도있는 국제 행사이다. 올림픽은 2년마다 하계 올림픽과 동계 올림픽이 번갈아 열리며, 국제 올림픽 위원회(IOC)가 감독하고 있다. 또한 오늘날의 올림픽은 기원전 8세기부터 서기 5세기에 이르기까지 고대 그리스 올림피아에서 열렸던 올림피아 제전에서 비롯되었다. 그리고 19세기 말에 피에르 드 쿠베르탱 남작이 고대 올림피아 제전에서 영감을 얻어, 근대 올림픽을 부활시켰다. 이를 위해 쿠베르탱 남작은 1894년에 IOC를 창설했으며, 2년 뒤인 1896년에 그리스 아테네에서 제 1회 올림픽이 열렸다. 이때부터 IOC는 올림픽 운동의 감독 기구가 되었으며, 조직과 활동은 올림픽 헌장을 따른다. 오늘날 전 세계 대부분의 국가에서 올림픽 메달은 매우 큰 영예이며, 특히 올림픽 금메달리스트는 국가 영웅급의 대우를 받으며 스포츠 스타가 된다. 국가별로 올림픽 메달리스트들에게 지급하는 포상금도 크다. 대부분의 인기있는 종목들이나 일상에서 쉽게 접하고 즐길 수 있는 생활스포츠 종목들이 올림픽이라는 한 대회에서 동시에 열리고, 전 세계 대부분의 국가 출신의 선수들이 참여하는 만큼 전 세계 스포츠 팬들이 가장 많이 시청하는 이벤트이다. 2008 베이징 올림픽의 모든 종목 누적 시청자 수만 47억 명에 달하며, 이는 인류 역사상 가장 많은 수의 인구가 시청한 이벤트였다.  \n또한 20세기에 올림픽 운동이 발전함에 따라, IOC는 변화하는 세계의 사회 환경에 적응해

# HyDE (Hypothetical Document Embedding)
## 개념
- 질문(query)에 대한 가상의 답변 문서를 생성하고, 이 생성된 가상의 답변 문서를 임베딩하여 검색에 활용하는 기법이다.
- 일반적인 RAG에서 검색은 질문을 임베딩하여 문서 임베딩과 직접 비교한다. 
- 질문("파리는 어떤 도시인가?")과 답변 문서("파리는 프랑스의 수도이며...")는 표현 방식이 다르다는 문제가 있다. 
- 즉 의미적으로는 관련있지만 벡터 공간(임베딩 벡터간의 유사서)에서는 거리가 멀 수가 있다.
- 그래서 HyDE는 질문과 문서가 아니라 질문으로 가상의 답변 문서를 만들고 **가상의 답변문서와 저장된 문서들간의 유사도**를 비교한다.

## HyDE 프로세스

1. 가상 문서 생성
   -  LLM을 사용해 질문에 대한 가상의 답변 문서 생성
   -  이때 성능이 좋은 LLM을 사용하는 것이 좋다.
2. 임베딩 변환
   - `1`에서 생성한 가상 문서를 벡터로 임베딩
3. 유사도 검색
   - 가상 문서 임베딩으로 Vector Store에 저장된 문서들 중 유사한 문서를 검색
4. 답변 생성
   - 검색된 실제 문서를 바탕으로 최종 답변 생성

## 장단점
- **장점**
  - 질문-문서 간 의미적 차이를 해결 해서 정확한 문서 검색 가능
  - 질문 표현 방식에 덜 민감
- **단점**
  - 가상 문서의 품질에 따른 성능 편차가 발생한다.
    - 가상 문서 생성 시 환각(hallucination) 위험
  - 추가적인 LLM 호출로 인한 비용 증가한다.

In [23]:
model = ChatOpenAI(model="gpt-5.5")
hyde_prompt = ChatPromptTemplate.from_template(
    template="""<instruction>
다음 질문에 대한 완전하고 상세한 답변을 사실에 기반하여 작성해 주세요.
질문과 관련된 내용으로 답변을 작성합니다.
질문과 직접적인 연관성이 없는 내용은 절대 포함시키지 않습니다.
답변은 1000글자 이내로 해주세요.
</instruction>
<question>
{query}
</question>
"""
)
hyde_chain = hyde_prompt | model | StrOutputParser()

In [24]:
query = "올림픽에서 발생한 다양한 논란들에 대해 정리해 줘."
result = hyde_chain.invoke({"query":query})

In [25]:
print(result)

올림픽 논란은 주로 다음 유형으로 정리된다.

1. **정치적 보이콧과 국제 갈등**  
   1936년 베를린 올림픽은 나치 독일의 선전장으로 비판받았다. 1980년 모스크바 대회는 소련의 아프가니스탄 침공에 항의해 미국 등 여러 나라가 불참했고, 1984년 LA 대회는 소련권 국가들이 보이콧했다.

2. **도핑 문제**  
   금지약물 사용은 반복된 논란이다. 1988년 서울 올림픽에서 벤 존슨은 도핑 적발로 금메달을 박탈당했다. 러시아는 국가 주도 도핑 의혹으로 2016년 이후 제재를 받았고, 일부 대회에서 국기와 국가 사용이 제한됐다.

3. **판정 논란**  
   체조, 복싱, 피겨스케이팅 등 채점 종목에서 편파 판정 의혹이 자주 제기됐다. 2002년 솔트레이크시티 피겨 페어 종목 판정 논란은 심판 제도 개편으로 이어졌다.

4. **인권·표현의 자유 문제**  
   개최국의 인권 상황, 소수자 탄압, 언론 통제 등이 논란이 됐다. 또한 선수들의 정치적 표현을 어디까지 허용할지에 대한 논쟁도 계속되고 있다.

5. **부패와 개최지 선정 의혹**  
   IOC 위원 매수 의혹 등 개최지 선정 과정의 투명성이 문제 된 사례가 있었다. 특히 2002년 솔트레이크시티 유치 과정에서 뇌물 스캔들이 드러났다.

6. **안전 문제와 테러**  
   1972년 뮌헨 올림픽에서는 팔레스타인 무장단체가 이스라엘 선수단을 인질로 잡아 11명이 사망하는 참사가 발생했다.

7. **경제·환경 논란**  
   막대한 개최 비용, 경기장 사후 활용 문제, 지역 주민 이주, 환경 훼손도 비판 대상이다. 최근에는 지속 가능한 올림픽 운영이 중요한 쟁점이 되고 있다.


In [26]:
# 전체 RAG cahin
# hyde

# PromptTemplate 구성
# 프롬프트 구성 - 질문 + 검색한문서들

prompt = ChatPromptTemplate.from_template(
    template="""<instruction>
당신은 친절한 QA Assistant입니다.
질문에 대해 주어진 context를 기반으로 답을 해주세요.
context에 질문과 관련된 내용이 없을 경우 "주어진 정보로는 답을 알 수 없습니다." 라고 답을 하세요.
context에 없는 내용으로 답을 만들지 마세요.
</instruction>
<context>
{context}
</context>
<question>
{query}
</question>
<output-format>
- 답변을 context의 어느부분을 참조하였는지 각주를 달아주세요.
- 답변을 markdown 형식으로 작성하세요.
- 답변 구성은 example을 참조하세요.
<example>
# 답변 
최종 답변내용

# 참조내용
context에서 참조한 내용
</example>
</output-format>"""
)
model = ChatOpenAI(model="gpt-5.4-mini")
parser = StrOutputParser()
def format_str(docs:list) -> str:
    """
    retriever가 찾은 문서 리스트(list[Document])에서 page_content만 추출해서 하나의 문자열로 합쳐서 반환하는 함수
    Prompt Template의 context에 넣어줄 값들만 str로 만든다.
    """
    return "\n\n".join(doc.page_content for doc in docs)

chain = {
    "context":hyde_chain | basic_retriever | format_str, 
    "query":RunnablePassthrough()
    
} | prompt | model | StrOutputParser()

In [27]:
response = chain.invoke(query)

In [28]:
print(response)

# 답변

올림픽에서는 여러 종류의 논란이 발생해 왔습니다.

1. **정치·체제 선전의 장으로 이용된 사례**
   - 올림픽은 쿠베르탱의 원래 이념과 달리 정치적 목적에 활용되기도 했습니다.  
   - 예를 들어 1936년 하계 올림픽에서는 나치독일이 자신들의 체제를 미화하고 아리안족의 우월함을 보여주려 했지만, 제시 오언스의 활약으로 그 의도는 실현되지 않았습니다.[^1]

2. **선수의 정치적 표현 문제**
   - 1968년 멕시코 시티 올림픽에서 토미 스미스와 존 카를로스는 시상식에서 블랙 파워 설루트를 했고, 피터 노먼도 이를 지지하는 의미로 OPHR 배지를 달았습니다.  
   - 이에 IOC 위원장이었던 에이버리 브런디지는 미국 올림픽 위원회에 두 선수의 귀국 조치를 요구했습니다.[^2]

3. **국가 차원의 보이콧**
   - 올림픽은 여러 차례 국가들의 보이콧 대상이 되었습니다.  
   - 1956년에는 소련의 헝가리 침공과 제2차 중동 전쟁을 이유로 일부 국가가 불참했고, 1972년·1976년에는 아프리카 국가들이 남아프리카 공화국과 로디지아의 인종차별정책에 항의해 보이콧했습니다.  
   - 1980년 모스크바 올림픽에서는 소련의 아프가니스탄 침공에 항의해 미국 등 65개국이 불참했고, 1984년 LA 올림픽에서는 소련과 동구권 14개국이 불참했습니다.  
   - 1976년에는 중화민국(타이완)도 중국의 압박에 반발해 보이콧했습니다.[^3]

4. **냉전과 국제정치의 영향**
   - 올림픽은 냉전의 영향도 크게 받았습니다.  
   - 소련은 1952년부터 본격적으로 참가해 이후 스포츠강국의 면모를 보였고, 1980년과 1984년에는 냉전 대립 속에서 서로 반대 진영의 올림픽에 불참하는 일이 발생했습니다.[^4]

5. **인권 문제와 외교 갈등**
   - 이란 정부는 이스라엘과의 경기 자체를 피하려 했고, 이란 선수들이 이스라엘 선수와 겨루는 상황에서 경기를 포기한 사례가 있었습니다.  
   - 2008년에는 중국의 티

#  MultiQueryRetriever

## 개념
- 하나의 사용자 질문으로 **여러 개의 다양한 질문을 생성하여 검색**을 수행하는 방법이다.
- 단일 질문의 한계를 극복하고 다각도에서 관련 정보 검색할 수있다.
  - 기본 RAG는 사용자의 질문의 질(quality)에 따라 검색 결과가 좌우된다.
  - 사용자가 한 질문에만 의존하는 것이 아니라 그 질문을 바탕으로 **다양한 의미의 질문들을 생성해서 단일 질문이 가지는 표현의 한계를 보완**한다.
    - 동일한 질문을 다른 각도에서 접근할 수있다.
    - 다양한 어휘와 표현으로 질문을 재구성한다.
- 예)
  - **원본 질문**: "딥러닝의 장점은 무엇인가?"
  - **생성된 질문들**:
    1. "딥러닝이 전통적인 머신러닝보다 나은 점은?"
    2. "딥러닝을 사용하면 얻을 수 있는 이익은?"
    3. "딥러닝의 주요 강점과 특징은?"
    4. "딥러닝 기술의 핵심 우위는?"
## 실행 프로세스

1. 질문 생성   
   - LLM을 사용해 원본 질문을 3-5개의 서로 다른 질문으로 변환
2. 병렬 검색
   - 생성된 각 질문으로 독립적으로 문서 검색 수행
3. 결과 통합
   - 여러 검색 결과를 하나로 병합
4. 중복 제거
   - 동일한 문서가 여러 번 검색된 경우 중복 제거
5. 최종 답변
   - 통합된 문서 세트를 바탕으로 답변 생성

## 장단점
- **장점**
    - 단일 질문으로 놓칠 수 있는 관련 문서 발견 수 있다.
    - 사용자 질문 표현 방식의 한계 극복
    - 더 포괄적이고 완전한 정보 검색 및 수집을 할 수있다.
- **단점**
    - 여러 번의 LLM 호출과 검색 수행이 실행 되므로 **계산비용, 토큰비용, 응답시간이 증가한다.**
    - 생성된 질문의 품질에 따른 성능 편차가 있을 수 있다.
    - 생성된 질문에 따라 원래 질문과 관련성 낮은 문서도 검색될 수 있어 최종 답변을 방해하는 노이즈가 증가할 수있다.

In [29]:
from langchain_classic.retrievers.multi_query import MultiQueryRetriever
mq_retriever = MultiQueryRetriever.from_llm(
    retriever=basic_retriever,
    llm=ChatOpenAI(model="gpt-5.4-mini"),
    # prompt=질문을 만들 때 사용할 PromptTemplate을 직접 넣어준다.
    
)

In [30]:
print(query)
dummy_result = mq_retriever.invoke(query)

올림픽에서 발생한 다양한 논란들에 대해 정리해 줘.


In [31]:
dummy_result

[Document(metadata={'Header 1': '올림픽', 'Header 2': '상업화', 'Header 3': '행사', '_id': '9d5ff1a5-d936-495b-bba6-4acb993c2a03', '_collection_name': 'olympic_info_wiki'}, page_content="올림픽에서 이루어지는 주요 행사로는 개막식, 폐막식, 시상식 등이 있다.  \n#### 개막식  \n개막식 때는 올림픽 헌장에 따라 다양한 행사가 열린다. 개막식의 기본 토대는 벨기에 안트베르펜에서 열린 1920년 하계 올림픽 때 만들어졌다. 개막식은 대개 개최국의 국기가 게양되고 국가가 울려퍼지며 시작된다. 그 후에 개최국이 준비한 그들의 문화를 대표하는 음악, 춤, 영상 따위가 공연된다. 개막식은 아름답기가 매회가 지날수록 웅대해지고 복잡해지는데, 이는 전 대회보다 사람들의 기억에 오래도록 남기기 위함이다. 보도에 의하면 2008 베이징 올림픽 개막식 때 든 비용은 1억 달러로 그 대부분이 예술적인 부분에 들었다고 한다.  \n행사가 끝나면 다음에는 각국의 선수단이 입장한다. 올림픽의 발상지라는 영예를 가진 그리스가 전통적으로 맨 처음에 입장한다. 나머지 각국 선수단은 주최국에서 선택한 언어의 사전 순으로 입장하고 나서, 개최국 선수단이 제일 마지막에 입장한다. 그리스 아테네에서 열린 2004년 하계 올림픽에서는 그리스 국기가 맨 처음에 입장하고, 그리스 선수단이 맨 마지막에 입장했다. 개막식 마지막에는 올림픽 성화가 들어오고 마지막 성화 봉송자에게 전달할 때까지 돌게 된다. 마지막 성화 봉송자는 대체로 유명하고 올림픽에서 성공한 개최국의 선수가 하며 올림픽 성화를 경기장 내에 점화한다. 그 다음 올림픽조직위원장과 IOC 위원장이 개막 선언을 하는데 공식적으로 개막되었다는 것과 올림픽 성화가 점화되는 것을 선언한다. 그 다음에 올림픽조직위원장과 IOC 위원장이 개회사를 낭독하게 된다. 마자막으로 올림픽기 게양에 이어 올림픽 선서를 끝으로 개막

In [32]:
mq_chain = {
    "context":mq_retriever | format_str,
    "query":RunnablePassthrough()
} | prompt | ChatOpenAI(model="gpt-5.4-mini") | StrOutputParser()

In [33]:
res = mq_chain.invoke(query)
print(res)

# 답변
올림픽에서 발생한 논란은 크게 다음과 같이 정리할 수 있습니다.

1. **전쟁과 정치 갈등으로 인한 올림픽 취소**
   - 제1차 세계대전과 제2차 세계대전 때문에 여러 차례 올림픽이 취소되었습니다.[^1]

2. **국가 간 무력 충돌과 올림픽 현안**
   - 2008년 베이징 올림픽 개막식 날 조지아와 러시아 간 전쟁이 발생했습니다.[^2]

3. **테러 사건**
   - 1972년 뮌헨 올림픽에서 검은 9월단의 테러로 이스라엘 선수들이 인질이 되었고, 1996년 애틀랜타 올림픽에서는 폭발 사건이 일어났습니다.[^3]

4. **도핑과 약물 복용 논란**
   - 초기에는 선수들이 기록 향상을 위해 약물을 사용했고, 이후 도핑 적발과 메달 박탈 사례가 이어졌습니다. 이를 계기로 WADA가 설립되고 도핑 검사 체계가 강화되었습니다.[^4]

5. **보이콧**
   - 1956년부터 여러 나라가 정치적 이유로 올림픽을 보이콧했고, 1980년과 1984년에는 냉전으로 인해 대규모 불참 사태가 있었습니다.[^5]

6. **정치 선전의 장으로 이용된 사례**
   - 1936년 나치독일은 올림픽을 체제 선전에 이용하려 했고, 1952년 이후 소련도 올림픽에서 정치적 의미를 강화했습니다.[^6]

7. **선수들의 정치적 표현**
   - 1968년 멕시코시티 올림픽에서 토미 스미스와 존 카를로스가 블랙 파워 설루트를 선보였고, 이에 대해 IOC가 강경 대응했습니다.[^7]

8. **IOC 내부의 부패·뇌물 논란**
   - IOC 위원들의 뇌물수수 의혹과 개최지 선정 과정의 부패 논란이 있었으며, 이에 대한 조사와 개혁도 이루어졌습니다.[^8]

9. **아마추어 정신과 프로 선수 참가 논란**
   - 올림픽은 원래 아마추어 정신을 중시했지만, 프로 선수 참가 허용 여부를 둘러싼 논란이 있었고 결국 종목별로 기준이 달라졌습니다.[^9]

# 참조내용
[^1]: “쿠베르탱의 생각과는 달리, 올림픽이 세계에 완벽한 평화를 가져다주지는 못했다. 실

# MapReduce RAG 방식

## 개요

- RAG(Retrieval-Augmented Generation)에서 검색된 문서들 중 질문과 관련성이 높은 문서만을 선별하여 더 정확한 답변을 생성하는 방법이다.
- 검색된 문서들 중에서 질문 답변에 실제로 도움이 되는 문서만을 LLM을 통해 선별한 후 전달하는 방식이다.

## MapReduce 방식 프로세스
1. Map (문서 검색)
   - 벡터스토어에서 질문과 유사한 문서들을 의미적 유사도 검색으로 찾는다.
   - 이 단계에서는 단순 벡터 유사도만 고려하므로 질문과 직접적인 관련이 없는 문서도 포함될 수 있다.
2. Reduce (문서 선별 및 요약)
   - 검색된 각 문서가 질문 답변에 실제로 도움이 되는지 LLM에게 평가 요청한다.
   - 관련성이 높은 문서들만 선별하여 요약하거나 결합한다.
   - 필요시 여러 문서의 정보를 통합하여 더 응답에 적합한 컨텍스트를 생성한다.
3.  Generate (최종 답변 생성)
    - 질문과 선별된 컨텍스트를 함께 LLM에 전달하여 최종 답변을 생성한다.

## 장단점

- **장점**
  - **높은 정확도**: 질문과 직접 관련된 정보만 사용하여 더 정확한 답변을 생성한다.
  - **노이즈 제거**: 유사하지만 관련 없는 정보로 인한 혼동을 방지한다.
  - **컨텍스트 최적화**: 제한된 토큰 범위 내에서 가장 유용한 정보만 전달한다.
  - **확장성**: 많은 문서가 검색되어도 중요한 정보만 선별하여 처리할 수 있다.
- **단점**
  - **추가 비용**: 문서 선별을 위한 LLM 호출로 인한 비용이 증가한다.
  - **처리 시간**: 문서 평가 단계가 추가되어 응답 속도가 저하된다.
  - **의존성**: 문서 선별 성능이 LLM의 판단 능력에 크게 의존한다.

In [34]:
# Reduce Chain: 질문과 문서간의 직접적인 연관성이 있는지 판단해서 연관된 문서만 반환한다.
reduce_prompt = ChatPromptTemplate.from_template(
    template="""<instrucftion>
Context의 내용이 질문과 관련성이 있으면 입력괸 context를 그대로 반환하고, 관련성이 없으면 아무것도 출력하지 않는다,
- 관련성 판단기준
    - Cintext에 질문에 대한 직접적인 답이나 답을 위한 단서가 있어야 한다.
    - Cintext의 내용이 질문을 해결하는데 도움을 줘야 한다.
</instrucftion>
<context>
{context}
</context>
<question>
{query}
</question>"""
)
reduce_chain = reduce_prompt | ChatOpenAI(model="gpt-5.4-mini") |StrOutputParser()

In [38]:
# context= "올림픽에는 300개의 종목이 있습니다."
# context= "사과는 과일입니다."
# context= "1회 올림픽은 그리스 아테네에서 열렸습니다."
context= "축구는 올림픽 종목에 포함되어 있습니다."
res = reduce_chain.invoke({"context":context, "query":"올림픽 종목에 대해 알려줘."})
print(res)

축구는 올림픽 종목에 포함되어 있습니다.


In [48]:
from langchain_core.runnables import chain
from langchain_core.documents import Document

@chain
def map_reduce(inputs:dict) -> str:
    """
    retriever가 검색한 문서들(list[Document])들과 사용자의 질문을 받아서 
    개별 문서와 질문간의 관련성을 reduce_chain을 이용해 검사한다.
    문서들중 reduce_chain을 통과한 내용들만 추려서 반환한다.
    Args:
        inputs(dict[str, Any]) -검색한 문서와 사용자 질문을 묶어서 dictionary로 받는다.
    Returns:
        str: 질문과 연관성 있는 문서들을 문자열로 묶어서 반환
    """
    
    docs:list[Document] = inputs['documents']
    query:str = inputs['query']
    context:str = ""
    for doc in docs:
        res = reduce_chain.invoke({"query":query, "context":doc.page_content})
        context += res+"\n\n"                
    return context.strip()

retriever = get_retriever(vectorstore, k=10)
map_reduce_chain = {
    "query": RunnablePassthrough(),
    "documents": retriever
} | map_reduce

In [49]:
res = map_reduce_chain.invoke("국제 올림픽 기구에 대해 설명해줘")

In [50]:
print(res)

올림픽 활동이란 많은 수의 국가, 국제 경기 연맹과 협회 • 미디어 파트너를 맺기 • 선수, 직원, 심판, 모든 사람과 기관이 올림픽 헌장을 지키는 것을 말한다.국제올림픽위원회(IOC)는 모든 올림픽 활동을 통솔하는 단체로서, 올림픽 개최 도시 선정, 계획 감독, 종목 변경, 스폰서 및 방송권 계약 체결 등의 권리가 있다. 올림픽 활동은 크게 세 가지로 구성된다.  
- 국제경기연맹(IF)은 국제적인 규모의 경기를 관리, 감독하는 기구이다. 예를 들어서 국제 축구 연맹(FIFA)는 축구를 주관하며, 국제 배구 연맹(FIVB)은 배구를 주관하는 기구이다. 올림픽에는 현재 35개의 국제경기연맹이 있고 각 종목을 대표한다. (이 중에는 올림픽 종목은 아니지만 IOC의 승인을 받은 연맹도 있다.)
- 국가 올림픽 위원회(NOC)는 각국의 올림픽 활동을 감독하는 기구이다. 예를 들어서 대한 올림픽 위원회(KOC)는 대한민국의 국가 올림픽 위원회이다. 현재 IOC에 소속된 국가 올림픽 위원회는 205개이다.
- 올림픽 조직 위원회(OCOG)는 임시적인 조직으로 올림픽의 총체적인 것(개막식, 페막식 등)을 책임지기 위해 구성된 조직이다. 올림픽 조직 위원회는 올림픽이 끝나면 해산되며 최종보고서를 IOC에 제출한다.  
올림픽의 공식언어는 프랑스어와 영어와 개최국의 공용어이다. 모든 선언(예를 들어서 개막식 때 각국 소개를 할 때)들은 세 언어가 모두 나오거나 영어나 프랑스어 중에서 한 언어로만 말하기도 한다. 개최국의 공용어가 영어나 프랑스어가 아닐 때는 당연히 그 나라의 공용어도 함께 나온다.



올림픽은 국제경기연맹(IF), 국가 올림픽 위원회(NOC), 각 올림픽의 위원회(예-벤쿠버동계올림픽조직위원회)로 구성된다. 의사 결정 기구인 IOC는 올림픽 개최 도시를 선정하며, 각 올림픽 대회마다 열리는 올림픽 종목도 IOC에서 결정한다. 올림픽 경기 개최 도시는 경기 축하 의식이 올림픽 헌장에 부합하도록 조직하고 기금을 마련해야 한다.

국제 올림픽 위원회(이하 IOC로

In [51]:
# 최종 Chain
chain = {
    "context":map_reduce_chain,
    "query":RunnablePassthrough()
} | prompt | ChatOpenAI(model="gpt-5.4-mini") |StrOutputParser()

In [52]:
response =chain.invoke("올림픽 경기 종목에 대해 설명해줘")

In [53]:
print(response)

# 답변
올림픽 경기 종목은 하계 올림픽과 동계 올림픽으로 나뉘며, 전체적으로는 **33개 부문 52개 종목, 약 400개의 경기**로 이루어져 있습니다.[^1]

하계 올림픽은 **26개 부문**, 동계 올림픽은 **7개 부문**으로 구성됩니다.[^2]  
하계 올림픽에서는 **육상, 수영, 펜싱, 체조**가 1회 대회부터 한 번도 빠짐없이 정식종목이었고, 동계 올림픽에서는 **크로스컨트리, 피겨 스케이팅, 아이스 하키, 노르딕 복합, 스키 점프, 스피드 스케이팅**이 1924년 동계 올림픽부터 빠짐없이 정식종목이었습니다.[^3]

또한 배드민턴, 농구, 배구처럼 처음에는 **시범종목**으로 시작했다가 이후 **정식종목**이 된 종목도 있습니다. 반대로 야구처럼 예전에는 정식종목이었지만 지금은 빠진 종목도 있습니다.[^4]

올림픽 종목은 IOC가 승인한 국제경기연맹의 관리를 받으며, 정식종목 선정은 IOC 총회에서 투표를 통해 이루어지고 **재적 위원 수의 과반수 이상 찬성**이 필요합니다.[^5]  
또한 올림픽 프로그램 위원회는 올림픽 종목 선정 기준으로 **역사, 전통, 보편성, 인기도와 잠재성, 선수의 건강, 연맹의 관리 능력, 개최 비용**의 7가지를 제시했습니다.[^6]

# 참조내용
[^1]: “올림픽 경기 종목은 총 33개 부문 52개 종목에서 약 400개의 경기로 이루어져 있다.”
[^2]: “하계 올림픽은 26개, 동계 올림픽은 7개 부문으로 이루어져있다.”
[^3]: “하계 올림픽에서는 육상, 수영, 펜싱, 체조가 1회 대회때부터 한번도 빠짐없이 정식종목이었으며, 동계 올림픽에서는 크로스컨트리, 피겨 스케이팅, 아이스 하키, 노르딕 복합, 스키 점프, 스피드 스케이팅이 1924년 동계 올림픽부터 빠짐없이 정식종목이었다.”
[^4]: “배드민턴, 농구, 배구와 같은 정식종목들은 처음에는 시범종목이었으며 그 후에 정식종목으로 승인 되었다. 야구처럼 예전에는 정식종목 이었지만 지금은 정식 종목에서 빠진 종목도 있다.”
[^5]: “각 올림픽 

# Self Query Retriever

- **Self-Query Retriever**는 사용자의 자연어 질문을 LLM을 통해 '의미 기반 검색 쿼리(semantic query)'와 '메타데이터 기반 필터 조건(metadata filter)'으로 자동 분해/구조화하여, 벡터 검색과 조건 기반 필터링을 동시에 수행하도록 설계된 Advanced RAG의 고급 Retriever 기법이다. 이를 통해 메타데이터의 속성을 정확히 반영한 정밀 검색이 가능하다.
- 즉 사용자 질의로 부터 메타데이터의 필터 조회시 사용할 값을 추출하여 metadata 조건 기반 필터링을 할 수 있도록 한다.

## Self Query Retriever의 구성 요소

- **LLM (Large Language Model)**
    - 자연어로 표현된 사용자 질문을 받아, 이를 메타데이터 조건과 필터링 조건이 포함된 정형 쿼리(Structured Query)로 변환하는 역할을 한다.
- **StructuredQuery**
    - 사용자의 질문을 기반으로 생성되는 구조화된 쿼리로 다음 두가지가 생성된다.
      - 벡터 유사도 검색을 위한 **의미 기반 검색 쿼리**(semantic query)
      - 문서 메타데이터를 이용한 **필터링 조건**(metadata filter)
  
- **Query Translator**
    - 생성된 StructuredQuery를 특정 벡터 데이터베이스(Qdrant, Chroma 등)의 쿼리 언어로 번역하여, 실제 검색이 가능하도록 한다.
- **Vector Database**
    - 변환된 쿼리를 바탕으로 벡터 유사도 검색과 메타데이터 조건 필터링을 함께 수행하여 관련 문서를 반환한다.

## Self Query Retriever 작동 원리

![selfquery retriever](figures/selfquery_retriever.jpg)

1. 사용자가 자연어 질의(Query)를 입력한다.
2. LLM이 입력된 자연어 질문을 해석하여 **Query Constructor**가 **StructuredQuery**로 변환한다.
3. **Query Translator**가 StructuredQuery를 **벡터 데이터베이스에서 이해할 수 있는 쿼리**로 변환한다.
4. 벡터 데이터베이스가 변환된 쿼리에 따라 문서를 검색한다.
5. 최종적으로, 검색 결과가 사용자에게 제공된다.

## 사용 예시
1. Query
    - "2023년에 발표된 OpenAI의 GPT 모델 관련 논문을 찾아줘."
2. Query Constructor가 위 질문을 다음과 같은 형태의 StructuredQuery로 변환한다.

    ```json
    {
        "query": "GPT 모델 논문",
        "filter": {
            "must": [
                {
                    "key": "year",
                    "match": {
                        "value": 2023
                    }
                },
                {
                    "key": "author",
                    "match": {
                        "value": "OpenAI"
                    }
                }
            ]
        }
    }
    ```
     - query: 벡터검색에 사용될 자연어 의미기반 쿼리(semantic query)
     - filter: 메타데이터 기반 필터 조건
3. 위의 StruncturedQuery는 Retriever와 연결된 Vector database의 검색 형식에 맞춰 query translator에 의해 변환 되고 이것을 이용해 검색을 진행한다. (형식은 DB 마다 다르다.)


## Self Query Retriever의 장점

- **정밀성**: 메타데이터 조건을 정확하게 지정하여 원하는 문서를 정밀하게 검색할 수 있다.
- **효율성**: 메타데이터 필터링을 통해 관련 없는 문서를 미리 제거하여 검색 대상 문서의 후보 집합을 사전에 축소함으로써, 벡터 유사도 계산 대상이 줄어들고 전체 검색 연산량을 감소시킬 수 있다.
- **사용자 편의성**: 사용자가 복잡한 쿼리 조건을 직접 작성하지 않고, 자연어로 간편하게 질문할 수 있다.

## 예제
- 필요 패키지 설치
  - pip install langchain langchain-community langchain-openai langchain-qdrant lark

> ### Lark 패키지
> 
> - Lark는 텍스트를 구조적으로 분석하기 위한 파싱 라이브러리로, 미리 정의한 문법에 따라 문장을 해석하여 정해진 형태(parse tree 또는 abstract syntax tree) 형태의 구조화된 결과를 생성한다.
> - 이 라이브러리는 컴파일러, 인터프리터, DSL(도메인 특화 언어), 쿼리 언어, 수식 해석기처럼 구조화된 입력을 처리해야 하는 다양한 프로그램에서 활용된다.
> - LangChain의 **SelfQueryRetriever**에서는 LLM이 생성한 쿼리 조건 문자열(output)을 석하여 메타데이터 필터 조건으로 변환하기 위해 Lark가 사용된다. Lark는 텍스트 형식의 조건을 분석하여 벡터 검색 시 사용되는 StructuredQuery의 filter(metadata filter) 구조로 변환해주는 역할을 담당한다.

In [ ]:
# ! uv pip install lark
# 커널 재시작

Resolved 1 package in 127ms
Prepared 1 package in 104ms
Installed 1 package in 24ms
 + lark==1.3.1


In [1]:
from langchain_qdrant import QdrantVectorStore
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams
from dotenv import load_dotenv

load_dotenv()

True

In [18]:
documents = [
    Document(
        page_content="과학자들이 공룡을 되살리고 대혼란이 일어난다",
        metadata={"title": "쥬라기 공원", "year": 1993, "rating": 7.7, "genre": "science fiction"},
    ),
    Document(
        page_content="다른 사람의 꿈속에 들어가 생각을 훔치는 '추출가'가, 반대로 타인의 무의식 속에 새로운 생각을 '심는(인셉션)' 작전을 수행하며 겪는 꿈과 현실의 경계에 관한 이야기.",
        metadata={"title": "인셉션", "year": 2010, "genre":"science fiction", "director": "Christopher Nolan", "rating": 8.2},
    ),
    Document(
        page_content="정신과 환자들을 위한 특별한 의료 기기를 개발한 치바 아스코 박사는 사람들의 꿈 안으로 접속해 무의식 속 요인을 찾아내 진료에 활용한다. 그러던 어느 날, 박사의 기기가 도둑을 맞게 되고, 악한 세력들에 맞서 사람들의 마음을 지키기 위한 모험이 펼쳐진다.",  # 쉼표 추가
        metadata={"title": "파프리카", "year": 2006, "genre": "animated", "director": "Satoshi Kon", "rating": 6.5},
    ),
    Document(
        page_content="19세기 미국 남북 전쟁 시기, 가난하지만 꿋꿋하고 우애 있게 살아가는 마치 가문의 네 자매(메그, 조, 베스, 에이미)가 겪는 사랑, 성장, 그리고 가족애를 그린 따뜻한 고전 소설을 영화화하였다.",
        metadata={"title": "작은 아씨들", "year": 2019, "director": "Greta Gerwig", "rating": 8.3},
    ),
    Document(
        page_content="장난감들이 살아 움직이며 신나게 즐긴다",
        metadata={"title": "토이 스토리", "year": 1995, "genre": "animated"},
    ),
    Document(
        page_content="금지된 구역(Zone) 안에 어떤 소원이든 이루어지는 방이 있다는 전설을 좇아, 안내인(스토커)이 작가와 교수를 이끌고 위험천만한 구역을 통과하는 철학적 여정을 그린 SF 스릴러",
        metadata={"title": "스토커", "year": 1979, "director": "Andrei Tarkovsky", "genre": "thriller","rating": 9.9},
    ),
]

In [2]:
# Vector Store 생성

COLLECTION_NAME = "example"
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

client = QdrantClient(host="localhost", port=6333)
# client.create_collection(
#     collection_name=COLLECTION_NAME,
#     vectors_config=VectorParams(size=1536, distance=Distance.COSINE)
# )

vectorstore = QdrantVectorStore(
    client=client,
    collection_name="example",
    embedding=embedding_model
)
# vectorstore.add_documents(documents=documents)

In [5]:
from langchain_classic.chains.query_constructor.schema import AttributeInfo
from langchain_classic.retrievers.self_query.base import SelfQueryRetriever
from langchain_openai import ChatOpenAI
from langchain_community.query_constructors.qdrant import QdrantTranslator

# payload 조건 filtering을 위해서 metadata에 대한 정보를 정의
metadata_fileds_schema = [
    AttributeInfo(
        name="title",
        description="영화제목",
        type="string",
    ),
    AttributeInfo(
        name="genre",
        description="영화장르. 다음 하나를 가진다. ['science fiction', 'comedy', 'thriller', 'romance', 'action', 'animated']",
        type="string",
    ),
    AttributeInfo(
        name="year",
        description="영화 개봉 연도",
        type="integer",
    ),
    AttributeInfo(
        name="rating",
        description="영화 평점. 1 - 10 사이의 실수값을 가진다",
        type="float",
    ),
    AttributeInfo(
        name="director",
        description="영화를 만든 감독의 영문 이름",
        type="string",
    ),
]
page_content_description = "영화에 대한 짧은 소개글."
llm = ChatOpenAI(model="gpt-5.4-mini")

sq_retriever = SelfQueryRetriever.from_llm(
    llm=llm,
    vectorstore=vectorstore,
    document_contents=page_content_description,
    metadata_field_info=metadata_fileds_schema,
    structured_query_translator=QdrantTranslator(metadata_key=vectorstore.metadata_payload_key),


)



In [6]:
vectorstore.metadata_payload_key, vectorstore.content_payload_key

('metadata', 'page_content')

In [8]:
docs = sq_retriever.invoke("평점이 8점 이상인 영화를 추천해줘")

In [ ]:
# print(docs)
docs

[Document(metadata={'title': '작은 아씨들', 'year': 2019, 'director': 'Greta Gerwig', 'rating': 8.3, '_id': 'f4cfcb64-2989-4337-84a1-abb66823734d', '_collection_name': 'example'}, page_content='19세기 미국 남북 전쟁 시기, 가난하지만 꿋꿋하고 우애 있게 살아가는 마치 가문의 네 자매(메그, 조, 베스, 에이미)가 겪는 사랑, 성장, 그리고 가족애를 그린 따뜻한 고전 소설을 영화화하였다.'), Document(metadata={'title': '인셉션', 'year': 2010, 'genre': 'science fiction', 'director': 'Christopher Nolan', 'rating': 8.2, '_id': 'a5785e3a-7b06-43f6-a0e8-6464a930074c', '_collection_name': 'example'}, page_content="다른 사람의 꿈속에 들어가 생각을 훔치는 '추출가'가, 반대로 타인의 무의식 속에 새로운 생각을 '심는(인셉션)' 작전을 수행하며 겪는 꿈과 현실의 경계에 관한 이야기."), Document(metadata={'title': '스토커', 'year': 1979, 'director': 'Andrei Tarkovsky', 'genre': 'thriller', 'rating': 9.9, '_id': '13f376fb-7f4d-4507-878f-b36c47f0948f', '_collection_name': 'example'}, page_content='금지된 구역(Zone) 안에 어떤 소원이든 이루어지는 방이 있다는 전설을 좇아, 안내인(스토커)이 작가와 교수를 이끌고 위험천만한 구역을 통과하는 철학적 여정을 그린 SF 스릴러')]


[Document(metadata={'title': '작은 아씨들', 'year': 2019, 'director': 'Greta Gerwig', 'rating': 8.3, '_id': 'f4cfcb64-2989-4337-84a1-abb66823734d', '_collection_name': 'example'}, page_content='19세기 미국 남북 전쟁 시기, 가난하지만 꿋꿋하고 우애 있게 살아가는 마치 가문의 네 자매(메그, 조, 베스, 에이미)가 겪는 사랑, 성장, 그리고 가족애를 그린 따뜻한 고전 소설을 영화화하였다.'),
 Document(metadata={'title': '인셉션', 'year': 2010, 'genre': 'science fiction', 'director': 'Christopher Nolan', 'rating': 8.2, '_id': 'a5785e3a-7b06-43f6-a0e8-6464a930074c', '_collection_name': 'example'}, page_content="다른 사람의 꿈속에 들어가 생각을 훔치는 '추출가'가, 반대로 타인의 무의식 속에 새로운 생각을 '심는(인셉션)' 작전을 수행하며 겪는 꿈과 현실의 경계에 관한 이야기."),
 Document(metadata={'title': '스토커', 'year': 1979, 'director': 'Andrei Tarkovsky', 'genre': 'thriller', 'rating': 9.9, '_id': '13f376fb-7f4d-4507-878f-b36c47f0948f', '_collection_name': 'example'}, page_content='금지된 구역(Zone) 안에 어떤 소원이든 이루어지는 방이 있다는 전설을 좇아, 안내인(스토커)이 작가와 교수를 이끌고 위험천만한 구역을 통과하는 철학적 여정을 그린 SF 스릴러')]

In [18]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_template(
    template="""<instruction>
당신은 영화 추천 전문 AI Assistant입니다.
주어진 Context를 바탕으로 사용자 질문에 답을 합니다.
Context에 영화가 없거나 질문과 관련된 내용이 없으면 "질문에 대한 정보가 부족합니다."라고 답을 합니다.
</instruction>
<context>
{context}
</context>
<question>
{query}
</question>
<output-format>
- 추천하는 영화의 제목(내용)과 metadata (개봉년도, 감독이름, 평점, 장르)를 함께 안내합니다.
- 한국어로 자연스러운 문장으로 답변합니다.
</output-format>
"""
)

def format_movies(documents:list[Document]) -> str:
    """검색된 문서들을 받아서 답변용 context 문자열로 변환해서 반환.
    list[Document] -> 각 문서의 page_content값과 metadata를 합쳐서 문자열로 반환
    """
    return "\n\n".join(f"내용{doc.page_content}, 영화정보: {doc.metadata}" for doc in documents)

rag_chain = {
    "context": sq_retriever | format_movies,
    "query": RunnablePassthrough()
} | prompt | llm | StrOutputParser()

In [20]:
query = "2000년 이후 개봉한 영화중에 평점이 8점 이상인 영화를 추천해 주세요."
resp = rag_chain.invoke(query)
resp
print(resp)

추천드릴 수 있는 영화는 다음과 같습니다.

1. **인셉션**  
   - 내용: 다른 사람의 꿈속에 들어가 생각을 훔치는 '추출가'가, 반대로 타인의 무의식 속에 새로운 생각을 심는 작전을 수행하며 겪는 꿈과 현실의 경계에 관한 이야기입니다.  
   - 메타데이터: **개봉년도 2010년, 감독 Christopher Nolan, 평점 8.2, 장르 science fiction**

2. **작은 아씨들**  
   - 내용: 19세기 미국 남북 전쟁 시기, 가난하지만 꿋꿋하고 우애 있게 살아가는 마치 가문의 네 자매가 겪는 사랑, 성장, 그리고 가족애를 그린 따뜻한 고전 소설을 영화화한 작품입니다.  
   - 메타데이터: **개봉년도 2019년, 감독 Greta Gerwig, 평점 8.3, 장르 정보 없음**


